In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils import class_weight

# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
df = pd.read_csv('../Email dataset/Clean_Proj_Dataset.csv')

# Ensure labels are 0/1 and float32
df['Email Type'] = df['Email Type'].astype('float32')

# ------------------------------------------------------------------
# 2. TextVectorization layer
# ------------------------------------------------------------------
max_words = 20_000
max_len   = 300

vectorizer = layers.TextVectorization(
    max_tokens=max_words,
    output_mode='int',
    output_sequence_length=max_len
)
vectorizer.adapt(df['clean'].values)

# ------------------------------------------------------------------
# 3. Create tf.data.Dataset (pure TensorFlow splitting & shuffling)
# ------------------------------------------------------------------
full_dataset = tf.data.Dataset.from_tensor_slices(
    (df['clean'].values, df['Email Type'].values)
)

# Shuffle once with a fixed seed
full_dataset = full_dataset.shuffle(
    buffer_size=len(df), seed=42, reshuffle_each_iteration=False
)

# 80/20 split
train_size = int(0.8 * len(df))

train_dataset = full_dataset.take(train_size)
val_dataset   = full_dataset.skip(train_size)

# Batch + prefetch for speed
BATCH_SIZE = 64
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset   = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ------------------------------------------------------------------
# 4. Compute class weights for imbalanced dataset
# ------------------------------------------------------------------
y = df['Email Type'].values
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced', classes=np.unique(y), y=y
)
class_weights_dict = dict(enumerate(class_weights_array))
print("Class weights:", class_weights_dict)

# ------------------------------------------------------------------
# 5. Model (100% Keras/TensorFlow)
# ------------------------------------------------------------------
model = models.Sequential([
    layers.Input(shape=(), dtype=tf.string),           # raw strings in
    vectorizer,                                        # turns text → integers
    layers.Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

model.summary()

# ------------------------------------------------------------------
# 6. Train
# ------------------------------------------------------------------
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    class_weight=class_weights_dict,   # <-- handle imbalance
    callbacks=[
        callbacks.EarlyStopping(patience=4, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ]
)

# ------------------------------------------------------------------
# 7. Predict on new emails
# ------------------------------------------------------------------
def predict(texts):
    if isinstance(texts, str):
        texts = [texts]
    probs = model.predict(texts, verbose=0)
    return ["SPAM" if p > 0.5 else "HAM" for p in probs.flatten()], probs.flatten()

# Example usage
texts = [
    "Free money!!! Click now to claim your prize!!!",
    "Hey, are we still meeting for lunch tomorrow?"
]

labels, confidences = predict(texts)
for t, l, c in zip(texts, labels, confidences):
    print(f"{l} ({c:.4f}) → {t}\n")

# ------------------------------------------------------------------
# 8. Save everything (model + vectorizer inside!)
# ------------------------------------------------------------------
model.save('spam_classifier_100percent_tf.keras')

W0000 00:00:1764073095.273011   23700 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0000 00:00:1764073095.640144   23700 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0000 00:00:1764073095.640410   23700 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0000 00:00:1764073095.640420   23700 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1764073095.642986   23700 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not us

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type float).